# 01 - Daten: VisDrone-DET 2019 vorbereiten

Download, COCO-Konvertierung, SAHI-Slicing (640x640 @ 20% Overlap),
Statistik und auto-generierter Report nach `docs/data_report.md`.

In [ ]:
# Zelle 0 - Imports
import json
import os
import subprocess
import sys
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
from PIL import Image
import matplotlib.patches as patches

In [ ]:
# Zelle 1 - Setup: Drive + Repo + Dependencies
!pip install -q mmengine pycocotools sahi gdown matplotlib Pillow

from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess

REPO = '/content/ba-mamba-neck'
if os.path.isdir(REPO):
    subprocess.run(['git', '-C', REPO, 'pull'], check=True)
else:
    subprocess.run(
        ['git', 'clone', 'https://github.com/raphaelkach/ba-mamba-neck.git', REPO],
        check=True,
    )
os.chdir(REPO)
sys.path.insert(0, REPO)

In [ ]:
# Zelle 2 - VisDrone herunterladen

RAW = Path('/content/visdrone/raw')
RAW.mkdir(parents=True, exist_ok=True)
DRIVE_SRC = Path('/content/drive/MyDrive/datasets/visdrone')

SPLITS = {
    'train': '1a2oHjcEcwXP8oUF95qiwrqzACb2YlUhn',
    'val':   '1bxK5zgLn0_L8x276eKkuYA_FzwCIjb59',
}

for split, file_id in SPLITS.items():
    zip_name = f'VisDrone2019-DET-{split}.zip'
    dst = RAW / zip_name
    extract_dir = RAW / f'VisDrone2019-DET-{split}'

    if extract_dir.exists() and any(extract_dir.iterdir()):
        print(f'{split} bereits entpackt - skip')
        continue

    drive_src = DRIVE_SRC / zip_name
    if drive_src.exists():
        print(f'{split}: kopiere von Drive...')
        !cp "{drive_src}" "{dst}"
    else:
        print(f'{split}: Download via gdown...')
        !gdown --fuzzy "https://drive.google.com/file/d/{file_id}/view" -O "{dst}"

    assert dst.exists() and dst.stat().st_size > 1_000_000, (
        f'{split}: Download fehlgeschlagen. '
        f'Lade manuell von https://github.com/VisDrone/VisDrone-Dataset '
        f'und lege als {dst} ab.'
    )

    print(f'{split}: entpacken...')
    !cd "{RAW}" && unzip -q -o "{zip_name}"

print("Done")

In [ ]:
# Zelle 3 - VisDrone konvertieren + SAHI-Slicing
!python data/prepare.py --output /content/visdrone

In [ ]:
# Zelle 4 - 5 Beispielbilder aus dem Val-Split mit Bounding Boxes
import random

# Zelle 4 - 5 Beispielbilder aus dem Val-Split mit Bounding Boxes
ROOT = Path('/content/visdrone')
VAL_JSON = ROOT / 'annotations' / 'val_unsliced.json'
VAL_IMAGES = ROOT / 'raw' / 'VisDrone2019-DET-val' / 'images'

coco = json.loads(VAL_JSON.read_text())
id_to_name = {c['id']: c['name'] for c in coco['categories']}
ann_by_img = {}
for a in coco['annotations']:
    ann_by_img.setdefault(a['image_id'], []).append(a)

random.seed(42)
samples = random.sample(coco['images'], k=5)

fig, axes = plt.subplots(5, 1, figsize=(12, 30))
for ax, img_info in zip(axes, samples):
    img = Image.open(VAL_IMAGES / img_info['file_name'])
    ax.imshow(img)
    for ann in ann_by_img.get(img_info['id'], []):
        x, y, w, h = ann['bbox']
        ax.add_patch(patches.Rectangle((x, y), w, h, linewidth=1,
                                       edgecolor='lime', facecolor='none'))
        ax.text(x, max(0, y - 2), id_to_name[ann['category_id']],
                color='lime', fontsize=6)
    ax.set_title(f"{img_info['file_name']} - {len(ann_by_img.get(img_info['id'], []))} objs")
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Zelle 5 - Datenanalyse
from eval.plot_style import apply_style, TEXTWIDTH
apply_style()

C_HIST   = '#0072B2'
C_HIST_L = '#E69F00'
C_SMALL  = '#E69F00'
C_ACCENT = '#D55E00'
C_TEXT   = '#333333'

ROOT = Path('/content/visdrone')
Path('results/figures').mkdir(parents=True, exist_ok=True)

splits = {
    'Train (original)':  ROOT / 'annotations' / 'train_unsliced.json',
    'Train (SAHI 640\u00b2)': ROOT / 'annotations' / 'train_sliced.json',
    'Val (original)':    ROOT / 'annotations' / 'val_unsliced.json',
    'Val (SAHI 640\u00b2)':   ROOT / 'annotations' / 'val_sliced.json',
}

def load_sides(path):
    coco = json.loads(path.read_text())
    areas = np.array([a['area'] for a in coco['annotations']])
    return np.sqrt(areas)


# --- Figure 1: Size distribution ---
fig, axes = plt.subplots(2, 2, figsize=(TEXTWIDTH, TEXTWIDTH * 0.7), sharex=True)

for ax, (label, path) in zip(axes.flat, splits.items()):
    sides = load_sides(path)
    n = len(sides)
    s = np.sum(sides < 32)
    m = np.sum((sides >= 32) & (sides < 96))
    l = np.sum(sides >= 96)

    ax.hist(sides, bins=100, color=C_HIST, alpha=0.8,
            edgecolor='none', linewidth=0)

    ax.axvline(32, color=C_SMALL, ls='--', lw=0.8, alpha=0.6)
    ax.axvline(96, color=C_ACCENT, ls='--', lw=0.8, alpha=0.6)

    for thresh, clr, lbl in [(32, C_SMALL, '32px'), (96, C_ACCENT, '96px')]:
        ax.annotate(lbl, xy=(thresh, 0), xycoords=('data', 'axes fraction'),
                    xytext=(thresh, -0.08), textcoords=('data', 'axes fraction'),
                    fontsize=7, color=clr, ha='center', va='top',
                    arrowprops=dict(arrowstyle='->', color=clr, lw=0.8))

    ax.text(0.97, 0.95,
            f'S {100*s/n:.0f}%  M {100*m/n:.0f}%  L {100*l/n:.0f}%',
            transform=ax.transAxes, fontsize=7, color=C_TEXT,
            va='top', ha='right', fontfamily='monospace')

    ax.set_xlim(0, 250)

axes[1, 0].set_xlabel('\u221aarea (px)')
axes[1, 1].set_xlabel('\u221aarea (px)')
axes[0, 0].set_ylabel('Annotations')
axes[1, 0].set_ylabel('Annotations')

for ax in axes.flat:
    ax.yaxis.set_major_formatter(
        ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

fig.savefig('results/figures/size_distribution.pdf')
plt.show()
print('-> results/figures/size_distribution.pdf')


# --- Figure 2: SAHI effect ---
fig, (ax_train, ax_val) = plt.subplots(1, 2,
    figsize=(TEXTWIDTH, TEXTWIDTH * 0.4), sharey=True)

size_labels = ['Small\n(<32\u00b2 px)', 'Medium\n(32\u00b2\u201396\u00b2 px)', 'Large\n(>96\u00b2 px)']
x = np.arange(3)
w = 0.32

for ax, split_label, orig_key, sahi_key in [
    (ax_train, 'Train', 'Train (original)', 'Train (SAHI 640\u00b2)'),
    (ax_val,   'Val',   'Val (original)',   'Val (SAHI 640\u00b2)'),
]:
    sides_orig = load_sides(splits[orig_key])
    sides_sahi = load_sides(splits[sahi_key])

    def pcts(sides):
        n = len(sides)
        return [
            100 * np.sum(sides < 32) / n,
            100 * np.sum((sides >= 32) & (sides < 96)) / n,
            100 * np.sum(sides >= 96) / n,
        ]

    p_orig = pcts(sides_orig)
    p_sahi = pcts(sides_sahi)

    bars_o = ax.bar(x - w/2, p_orig, w, label='Original',
                    color=C_HIST, alpha=0.82, edgecolor='none')
    bars_s = ax.bar(x + w/2, p_sahi, w, label='SAHI 640\u00b2',
                    color=C_HIST_L, alpha=0.82, edgecolor='none')

    for bars in [bars_o, bars_s]:
        for bar in bars:
            h = bar.get_height()
            if h > 3:
                ax.text(bar.get_x() + bar.get_width()/2, h + 1,
                        f'{h:.0f}', ha='center', va='bottom',
                        fontsize=7, color=C_TEXT)

    for i in range(3):
        delta = p_sahi[i] - p_orig[i]
        if abs(delta) > 2:
            mid_y = max(p_orig[i], p_sahi[i]) + 6
            sign = '+' if delta > 0 else ''
            ax.text(x[i], mid_y, f'{sign}{delta:.0f}pp',
                    ha='center', va='bottom', fontsize=6,
                    color=C_ACCENT, fontweight='bold')

    ax.set_xticks(x)
    ax.set_xticklabels(size_labels)
    ax.set_title(split_label, fontsize=10)
    ax.set_ylim(0, 100)

ax_train.set_ylabel('Share (%)')
ax_val.legend(fontsize=8, loc='upper right')

fig.savefig('results/figures/sahi_effect.pdf')
plt.show()
print('-> results/figures/sahi_effect.pdf')


# --- Figure 3: Class distribution ---
fig, ax = plt.subplots(figsize=(TEXTWIDTH, TEXTWIDTH * 0.45))

coco_train = json.loads(
    (ROOT / 'annotations' / 'train_unsliced.json').read_text())
id_to_name = {c['id']: c['name'] for c in coco_train['categories']}
cls_counts = defaultdict(int)
for a in coco_train['annotations']:
    cls_counts[id_to_name[a['category_id']]] += 1

sorted_cls = sorted(cls_counts.items(), key=lambda x: x[1])
names = [c[0] for c in sorted_cls]
counts = [c[1] for c in sorted_cls]
max_count = max(counts)

bars = ax.barh(names, counts, color=C_HIST, alpha=0.8,
               height=0.6, edgecolor='white', linewidth=0.3)
ax.grid(axis='x', alpha=0.15, linewidth=0.5)
ax.set_axisbelow(True)

for bar, count in zip(bars, counts):
    ax.text(count + max_count * 0.015,
            bar.get_y() + bar.get_height()/2,
            f'{count:,}', va='center', fontsize=7, color=C_TEXT)

ax.set_xlabel('Annotations (train, unsliced)')
ax.set_xlim(0, max_count * 1.15)
ax.xaxis.set_major_formatter(
    ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

fig.savefig('results/figures/class_distribution.pdf')
plt.show()
print('-> results/figures/class_distribution.pdf')

In [ ]:
# Zelle 6 - docs/data_report.md generieren (Skript ist single source of truth)
!PYTHONPATH=. python scripts/generate_data_report.py --visdrone /content/visdrone

In [ ]:
# Zelle - Vorbereitete Daten auf Drive sichern
import shutil

DRIVE_DATA = Path('/content/drive/MyDrive/ba/visdrone_prepared')
SRC = Path('/content/visdrone')

if not DRIVE_DATA.exists():
    print('Kopiere vorbereitete Daten auf Drive (einmalig)...')
    shutil.copytree(SRC / 'annotations', DRIVE_DATA / 'annotations')
    shutil.copytree(SRC / 'sliced', DRIVE_DATA / 'sliced')
    shutil.copytree(SRC / 'raw', DRIVE_DATA / 'raw')
    print(f'Done: {DRIVE_DATA}')
else:
    print('Daten bereits auf Drive vorhanden.')

In [ ]:
# Zelle 7 - Figures auf Google Drive kopieren
import shutil

DRIVE_FIG = Path('/content/drive/MyDrive/ba/figures')
DRIVE_FIG.mkdir(parents=True, exist_ok=True)

for pdf in Path('results/figures').glob('*.pdf'):
    shutil.copy(pdf, DRIVE_FIG / pdf.name)
    print(f'-> {pdf.name}')

print(f'\nAlle Figures unter {DRIVE_FIG}')